<a href="https://colab.research.google.com/github/maruson08/new-folder-3/blob/main/Music_Generating_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchaudio numpy soundfile librosa tqdm pyyaml


In [ ]:
import glob
DATA_ROOT = "/content/mp3s"
SAVE_DIR  = "/content/mel_diffusion"
glob.glob(DATA_ROOT + "/*.mp3")

['/content/mp3s/TroyBoi - Do You_.mp3',
 '/content/mp3s/Kotori - Eroge.mp3',
 '/content/mp3s/No One.mp3']

In [ ]:
# ============================
# 0. Colab 환경 준비
# ============================
!nvidia-smi
!pip install torch torchaudio librosa tqdm soundfile pyyaml

import os, glob, random, math
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio, librosa, numpy as np
import soundfile as sf
from tqdm import tqdm
import IPython.display as ipd

assert torch.cuda.is_available(), "런타임 → GPU 로 설정하세요!"

# ============================
# 1. 데이터셋 정의 (완전 고정 길이 멜)
# ============================
class MP3Dataset(Dataset):
    def __init__(self, root, sr=44100, sec=5, n_fft=1024, hop=256, n_mels=80):
        self.files = glob.glob(os.path.join(root, "*.mp3"))
        self.sr, self.sec = sr, sec
        self.n_fft, self.hop, self.n_mels = n_fft, hop, n_mels
        self.length = sr * sec
        self.expected_frames = math.ceil((self.length + n_fft) / hop)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        wav, sr = torchaudio.load(path)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(0)  # mono

        # 길이 맞춤
        target_len = self.length + self.n_fft
        if len(wav) < target_len:
            wav = F.pad(wav, (0, target_len - len(wav)))
        else:
            start = random.randint(0, len(wav) - target_len)
            wav = wav[start:start + target_len]

        # 멜 스펙트로그램
        mel = librosa.feature.melspectrogram(
            y=wav.numpy(), sr=self.sr, n_fft=self.n_fft,
            hop_length=self.hop, n_mels=self.n_mels
        )
        mel = librosa.power_to_db(mel).astype(np.float32)

        # 정확히 expected_frames로 맞춤
        mel = mel[:, :self.expected_frames]
        if mel.shape[1] < self.expected_frames:
            pad_width = self.expected_frames - mel.shape[1]
            mel = np.pad(mel, ((0,0),(0,pad_width)), mode='constant')

        return torch.tensor(mel)

# ============================
# 2. UNet 모델 (출력 크기 보정 포함)
# ============================
class UNet(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.enc1 = nn.Conv2d(1, channels, 3, padding=1)
        self.enc2 = nn.Conv2d(channels, channels*2, 3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(channels*2, channels*4, 3, stride=2, padding=1)
        self.dec1 = nn.ConvTranspose2d(channels*4, channels*2, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(channels*2, channels, 4, stride=2, padding=1)
        self.outc = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x):
        h1 = F.relu(self.enc1(x))
        h2 = F.relu(self.enc2(h1))
        h3 = F.relu(self.enc3(h2))
        d1 = F.relu(self.dec1(h3))
        d2 = F.relu(self.dec2(d1))
        out = self.outc(d2)

        # 입력 크기와 동일하게 맞춤
        if out.shape[3] > x.shape[3]:
            out = out[:, :, :, :x.shape[3]]
        elif out.shape[3] < x.shape[3]:
            pad_width = x.shape[3] - out.shape[3]
            out = F.pad(out, (0,pad_width))

        return out

# ============================
# 3. Griffin-Lim 복원
# ============================
def griffin_lim(mel, sr=44100, n_fft=1024, hop=256, n_iter=32):
    mel = librosa.db_to_power(mel)
    S = librosa.feature.inverse.mel_to_stft(mel, sr=sr, n_fft=n_fft)
    wav = librosa.griffinlim(S, hop_length=hop, n_iter=n_iter)
    return wav

# ============================
# 4. 학습 함수
# ============================
def train_model(data_root, save_path, sample_rate=44100, sec=5, batch_size=2, epochs=2):
    ds = MP3Dataset(data_root, sr=sample_rate, sec=sec)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)
    model = UNet().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        for mel in tqdm(dl, desc=f"Epoch {epoch}"):
            mel = mel.unsqueeze(1).cuda()  # [B,1,F,T]
            noise = torch.randn_like(mel)
            out = model(noise)
            loss = F.mse_loss(out, mel)
            opt.zero_grad(); loss.backward(); opt.step()
        torch.save(model.state_dict(), save_path)
    print("✅ Training done.")
    return model

# ============================
# 5. 샘플 생성 함수
# ============================
def generate_sample(model, sec=5, sample_rate=44100, out_path="sample.wav"):
    with torch.no_grad():
        expected_frames = math.ceil((sec*sample_rate + 1024) / 256)
        z = torch.randn(1,1,80, expected_frames).cuda()
        mel = model(z)[0,0].cpu().numpy()
        wav = griffin_lim(mel, sr=sample_rate)
        sf.write(out_path, wav, sample_rate)
        print(f"✅ Saved {out_path}")
        return out_path

# ============================
# 6. 실행 예시
# ============================
DATA_ROOT = "/content/mp3s"
SAVE_DIR  = "/content/mel_diffusion"
os.makedirs(SAVE_DIR, exist_ok=True)

# 학습
model = train_model(DATA_ROOT, f"{SAVE_DIR}/ckpt_last.pt", epochs=1)

# 샘플 생성
sample_path = generate_sample(model, sec=5, out_path=f"{SAVE_DIR}/out_0000.wav")

# Colab에서 재생
ipd.Audio(sample_path)


Wed Aug 20 12:16:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             29W /   70W |     666MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/

✅ Training done.
✅ Saved /content/mel_diffusion/out_0000.wav


In [ ]:
# ============================
# 0. Colab 환경 준비
# ============================
!nvidia-smi
!pip install torch torchaudio librosa tqdm soundfile pyyaml

import os, glob, random, math
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio, librosa, numpy as np
import soundfile as sf
from tqdm import tqdm
import IPython.display as ipd

assert torch.cuda.is_available(), "런타임 → GPU 로 설정하세요!"
device = 'cuda'

# ============================
# 1. MP3Dataset (1분 이상)
# ============================
class MP3Dataset(Dataset):
    def __init__(self, root, sr=44100, sec=60, n_fft=1024, hop=256, n_mels=80):
        self.files = glob.glob(os.path.join(root, "*.mp3"))
        self.sr, self.sec = sr, sec
        self.n_fft, self.hop, self.n_mels = n_fft, hop, n_mels
        self.length = sr * sec
        self.expected_frames = math.ceil((self.length + n_fft) / hop)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        wav, sr = torchaudio.load(path)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(0)

        target_len = self.length + self.n_fft
        if len(wav) < target_len:
            wav = F.pad(wav, (0, target_len - len(wav)))
        else:
            start = random.randint(0, len(wav) - target_len)
            wav = wav[start:start + target_len]

        mel = librosa.feature.melspectrogram(
            y=wav.numpy(), sr=self.sr, n_fft=self.n_fft,
            hop_length=self.hop, n_mels=self.n_mels
        )
        mel = np.log(np.maximum(mel, 1e-5)).astype(np.float32)  # log-power
        mel = mel[:, :self.expected_frames]
        if mel.shape[1] < self.expected_frames:
            mel = np.pad(mel, ((0,0),(0,self.expected_frames - mel.shape[1])), mode='constant')

        return torch.tensor(mel)

# ============================
# 2. Diffusion UNet
# ============================
class DiffUNet(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.enc1 = nn.Conv2d(1, channels, 3, padding=1)
        self.enc2 = nn.Conv2d(channels, channels*2, 3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(channels*2, channels*4, 3, stride=2, padding=1)
        self.dec1 = nn.ConvTranspose2d(channels*4, channels*2, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(channels*2, channels, 4, stride=2, padding=1)
        self.outc = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x):
        h1 = F.relu(self.enc1(x))
        h2 = F.relu(self.enc2(h1))
        h3 = F.relu(self.enc3(h2))
        d1 = F.relu(self.dec1(h3))
        d2 = F.relu(self.dec2(d1))
        out = self.outc(d2)
        if out.shape[3] > x.shape[3]:
            out = out[:, :, :, :x.shape[3]]
        elif out.shape[3] < x.shape[3]:
            out = F.pad(out, (0,x.shape[3]-out.shape[3]))
        return out

# ============================
# 3. Cosine schedule & noise
# ============================
def cosine_schedule(t):
    return math.cos(t*math.pi/2)**2

def add_noise(x, t):
    alpha = cosine_schedule(t)
    noise = torch.randn_like(x)
    x_noisy = x * math.sqrt(alpha) + noise * math.sqrt(1-alpha)
    return x_noisy, noise, alpha

# ============================
# 4. Training
# ============================
def train_diffusion(data_root, save_path, sr=44100, sec=60, batch_size=1, epochs=5):
    ds = MP3Dataset(data_root, sr=sr, sec=sec)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)
    model = DiffUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        for mel in tqdm(dl, desc=f"Epoch {epoch}"):
            mel = mel.unsqueeze(1).to(device)
            t = random.random()
            x_noisy, noise, _ = add_noise(mel, t)
            pred = model(x_noisy)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
        torch.save(model.state_dict(), save_path)
    print("✅ Training done.")
    return model

# ============================
# 5. Sampling (1분 이상)
# ============================
def generate_diffusion(model, sr=44100, sec=60, steps=100, out_path="sample.wav"):
    expected_frames = math.ceil((sec*sr + 1024)/256)
    z = torch.randn(1,1,80,expected_frames).to(device)
    with torch.no_grad():
        for i in reversed(range(steps)):
            t = i / steps
            alpha = cosine_schedule(t)
            alpha_t = torch.tensor(alpha, device=device)
            noise_pred = model(z)
            z = (z - (1 - alpha_t)/torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            z = torch.tanh(z)  # 안정화

    mel = z[0,0].cpu().numpy()
    power = np.exp(mel)  # log-power → linear
    power = np.nan_to_num(power)  # NaN/Inf 제거
    wav = librosa.griffinlim(power, hop_length=256, n_iter=128)
    sf.write(out_path, wav, sr)
    print(f"✅ Saved {out_path}")
    return out_path

# ============================
# 6. 실행
# ============================
DATA_ROOT = "/content/mp3s"
SAVE_DIR  = "/content/mel_diffusion"
os.makedirs(SAVE_DIR, exist_ok=True)

# 학습
model = train_diffusion(DATA_ROOT, f"{SAVE_DIR}/ckpt_last.pt", epochs=5)  # 5 epoch 이상 권장

# 1분 음악 생성
sample_path = generate_diffusion(model, sec=60, out_path=f"{SAVE_DIR}/out_1min.wav")
ipd.Audio(sample_path)


Wed Aug 20 12:30:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             30W /   70W |    1718MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Epoch 0:   0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/

✅ Training done.
✅ Saved /content/mel_diffusion/out_1min.wav


In [ ]:
!pip install hifi_gan

  Preparing metadata (setup.py) ... done
  Created wheel for hifi_gan: filename=hifi_gan-0.0.2-py3-none-any.whl size=14172 sha256=bdec83f68d67917f66c0933b3c8f731d246fb2ee65ca79e31a348323681ea1af
  Stored in directory: /root/.cache/pip/wheels/01/63/62/7125e79d4cede44833d9d594bf585685ed2f0ae51dc5708f11
Successfully built hifi_gan


In [ ]:
!git clone https://github.com/jik876/hifi-gan.git
%cd hifi-gan
!pip install -r requirements.txt



Cloning into 'hifi-gan'...
remote: Enumerating objects: 48, done.
remote: Total 48 (delta 0), reused 0 (delta 0), pack-reused 48 (from 1)
Receiving objects: 100% (48/48), 620.94 KiB | 11.29 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/hifi-gan
ERROR: Could not find a version that satisfies the requirement torch==1.4.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==1.4.0


In [ ]:
# ============================
# 0. Colab 환경 준비
# ============================
!nvidia-smi
!pip install torch torchaudio librosa soundfile tqdm git+https://github.com/jik876/hifi-gan.git

import os, glob, random, math
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio, librosa, numpy as np
import soundfile as sf
from tqdm import tqdm
import IPython.display as ipd
from hifi_gan import Generator

device = 'cuda'
assert torch.cuda.is_available(), "런타임 → GPU 로 설정하세요!"

# ============================
# 1. MP3Dataset (1분 이상)
# ============================
class MP3Dataset(Dataset):
    def __init__(self, root, sr=44100, sec=60, n_fft=1024, hop=256, n_mels=80):
        self.files = glob.glob(os.path.join(root, "*.mp3"))
        self.sr, self.sec = sr, sec
        self.n_fft, self.hop, self.n_mels = n_fft, hop, n_mels
        self.length = sr * sec
        self.expected_frames = math.ceil((self.length + n_fft) / hop)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        wav, sr = torchaudio.load(path)
        if sr != self.sr:
            wav = torchaudio.functional.resample(wav, sr, self.sr)
        wav = wav.mean(0)

        target_len = self.length + self.n_fft
        if len(wav) < target_len:
            wav = F.pad(wav, (0, target_len - len(wav)))
        else:
            start = random.randint(0, len(wav) - target_len)
            wav = wav[start:start + target_len]

        mel = librosa.feature.melspectrogram(
            y=wav.numpy(), sr=self.sr, n_fft=self.n_fft,
            hop_length=self.hop, n_mels=self.n_mels
        )
        mel = np.log(np.maximum(mel, 1e-5)).astype(np.float32)  # log-power
        mel = mel[:, :self.expected_frames]
        if mel.shape[1] < self.expected_frames:
            mel = np.pad(mel, ((0,0),(0,self.expected_frames - mel.shape[1])), mode='constant')

        return torch.tensor(mel)

# ============================
# 2. Diffusion UNet
# ============================
class DiffUNet(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        self.enc1 = nn.Conv2d(1, channels, 3, padding=1)
        self.enc2 = nn.Conv2d(channels, channels*2, 3, stride=2, padding=1)
        self.enc3 = nn.Conv2d(channels*2, channels*4, 3, stride=2, padding=1)
        self.dec1 = nn.ConvTranspose2d(channels*4, channels*2, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(channels*2, channels, 4, stride=2, padding=1)
        self.outc = nn.Conv2d(channels, 1, 3, padding=1)

    def forward(self, x):
        h1 = F.relu(self.enc1(x))
        h2 = F.relu(self.enc2(h1))
        h3 = F.relu(self.enc3(h2))
        d1 = F.relu(self.dec1(h3))
        d2 = F.relu(self.dec2(d1))
        out = self.outc(d2)
        if out.shape[3] > x.shape[3]:
            out = out[:, :, :, :x.shape[3]]
        elif out.shape[3] < x.shape[3]:
            out = F.pad(out, (0,x.shape[3]-out.shape[3]))
        return out

# ============================
# 3. Cosine schedule & noise
# ============================
def cosine_schedule(t):
    return math.cos(t*math.pi/2)**2

def add_noise(x, t):
    alpha = cosine_schedule(t)
    noise = torch.randn_like(x)
    x_noisy = x * math.sqrt(alpha) + noise * math.sqrt(1-alpha)
    return x_noisy, noise, alpha

# ============================
# 4. Training
# ============================
def train_diffusion(data_root, save_path, sr=44100, sec=60, batch_size=1, epochs=5):
    ds = MP3Dataset(data_root, sr=sr, sec=sec)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)
    model = DiffUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        for mel in tqdm(dl, desc=f"Epoch {epoch}"):
            mel = mel.unsqueeze(1).to(device)
            t = random.random()
            x_noisy, noise, _ = add_noise(mel, t)
            pred = model(x_noisy)
            loss = F.mse_loss(pred, noise)
            opt.zero_grad(); loss.backward(); opt.step()
        torch.save(model.state_dict(), save_path)
    print("✅ Training done.")
    return model

# ============================
# 5. HiFi-GAN vocoder 로드
# ============================
vocoder = Generator().to(device)
state_dict = torch.hub.load_state_dict_from_url(
    'https://github.com/jik876/hifi-gan/releases/download/v1.0/generator_v1',
    map_location=device
)
vocoder.load_state_dict(state_dict['generator'])
vocoder.eval()

def mel_to_wave_hifi(mel, sr=44100, out_path="sample.wav"):
    mel_tensor = torch.tensor(mel).unsqueeze(0).to(device)
    with torch.no_grad():
        wav = vocoder(mel_tensor)
    wav = wav.squeeze().cpu().numpy()
    sf.write(out_path, wav, sr)
    print(f"✅ Saved {out_path}")
    return out_path

# ============================
# 6. Sampling
# ============================
def generate_diffusion(model, sr=44100, sec=60, steps=100, out_path="sample.wav"):
    expected_frames = math.ceil((sec*sr + 1024)/256)
    z = torch.randn(1,1,80,expected_frames).to(device)
    with torch.no_grad():
        for i in reversed(range(steps)):
            t = i / steps
            alpha = cosine_schedule(t)
            alpha_t = torch.tensor(alpha, device=device)
            noise_pred = model(z)
            z = (z - (1 - alpha_t)/torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)
            z = torch.tanh(z)  # 값 안정화

    mel = z[0,0].cpu().numpy()
    return mel_to_wave_hifi(mel, sr=sr, out_path=out_path)

# ============================
# 7. 실행
# ============================
DATA_ROOT = "/content/mp3s"
SAVE_DIR  = "/content/mel_diffusion"
os.makedirs(SAVE_DIR, exist_ok=True)

# 학습
model = train_diffusion(DATA_ROOT, f"{SAVE_DIR}/ckpt_last.pt", epochs=5)  # 최소 5 epoch

# 1분 음악 생성
sample_path = generate_diffusion(model, sec=60, out_path=f"{SAVE_DIR}/out_1min_hifi.wav")
ipd.Audio(sample_path)


Wed Aug 20 12:39:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             30W /   70W |    1718MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ImportError: cannot import name 'Generator' from 'hifi_gan' (unknown location)